# Import dependencies

In [26]:
from ipyleaflet import Map, basemaps, GeoJSON, Choropleth, FullScreenControl, ZoomControl, WidgetControl
from ipywidgets import HTML, Button, Dropdown, IntSlider, HBox, VBox, Label, Layout, RadioButtons
from branca.colormap import linear
import plotly.graph_objects as go
import geopandas as gpd
import pandas as pd
import json

# Loading Mapping Data

In [27]:
#LSOA to MSOA mapping table
LSOA_to_MSOA = pd.read_csv("OA_to_LSOA_to_MSOA_to_LAD_(December 2021).csv", usecols=["LSOA21CD", "LSOA21NM", "MSOA21CD", "MSOA21NM"]).drop_duplicates()

#MSOA to LAD mapping table
MSOA_to_LAD = pd.read_csv("OA_to_LSOA_to_MSOA_to_LAD_(December 2021).csv", usecols=["MSOA21CD", "MSOA21NM", "LAD22CD", "LAD22NM"]).drop_duplicates().rename(columns={"LAD22CD":"LAD21CD", "LAD22NM":"LAD21NM"})

#LAD to PFA mapping table
LAD_to_PFA = pd.read_excel("LAD_to_PFA_(December 2021).xlsx", usecols=["LAD21CD", "LAD21NM", "PFA21CD", "PFA21NM"]).drop_duplicates()

#Merging the previous two mapping tables
MSOA_to_PFA = pd.merge(MSOA_to_LAD, LAD_to_PFA, on="LAD21CD", how="inner").drop(["LAD21CD", "LAD21NM_x", "LAD21NM_y"], axis=1)

#Allocation buckets to Crime type data
data = {"alloc_bucket": ["Neighbourhood Problem-Solving and Reassurance", 
                         "Acquisitive and Place-Based Prevention", "Acquisitive and Place-Based Prevention", "Acquisitive and Place-Based Prevention", "Acquisitive and Place-Based Prevention", 
                         "Harm, Disruption and Enforcement", "Harm, Disruption and Enforcement"],
        "crime_type": ["Anti-social behaviour", 
                       "Burglary", "Vehicle crime", "Criminal damage and arson", "Property theft", 
                       "Violence", "Disruptive crimes"]
       }
bucket_to_crime = pd.DataFrame(data)

# Loading Geospatial Data

In [28]:
#Boundary data of PFAs
with open("PFA_(2021)_BGC.geojson",'r') as pfa:
    pfa_data = json.load(pfa)
with open("ENG_pfa_data.geojson",'r') as pfa:
    ENG_pfa_data = json.load(pfa)
    
#Boundary data of MSOAs
with open("MSOA_(2021)_BGC.geojson",'r') as msoa:
    msoa_data = json.load(msoa)
with open("ENG_msoa_data.geojson",'r') as msoa:
    ENG_msoa_data = json.load(msoa)

#Rename id values in the geojsons to the PFA or MSOA codes for easier handling
for i in pfa_data['features']:
    i['id'] = i['properties']['PFA21CD']
for i in msoa_data['features']:
    i['id'] = i['properties']['MSOA21CD']
for i in ENG_pfa_data['features']:
    i['id'] = i['properties']['PFA21CD']
for i in ENG_msoa_data['features']:
    i['id'] = i['properties']['MSOA21CD']

#English MSOA demographics data
ENG_MSOA_demog_data = pd.read_csv("msoa_data/ENG_MSOA_demographic_data.csv")

#English PFA to FTE
ENG_PFA_FTEs = pd.read_csv("force_msoa_resource_summary_v1.csv", usecols=["pfa_code", "msoa_count", "total_capacity", "pcso_fte", "regular_police_staff_fte"]).drop_duplicates().rename(columns={"pfa_code":"PFA21CD", "total_capacity":"total_fte", "regular_police_staff_fte":"officer_fte"})

#English Allocation
ENG_MSOA_alloc = pd.read_csv("final_allocation_v1.csv", usecols=["pfa_code", "msoa_code", "month", "bucket", "total_alloc", "uncertainty_flag"]).drop_duplicates().rename(columns={"pfa_code":"PFA21CD", "msoa_code":"MSOA21CD", "uncertainty_flag":"uncertain"})

In [29]:
'''
c=0
for i in ENG_pfa_data["features"]:
    if i["id"].startswith("W"):
        print(c)
        ENG_pfa_data["features"].pop(c)
    c += 1
len(ENG_pfa_data["features"]), len(pfa_data["features"])
'''

'\nc=0\nfor i in ENG_pfa_data["features"]:\n    if i["id"].startswith("W"):\n        print(c)\n        ENG_pfa_data["features"].pop(c)\n    c += 1\nlen(ENG_pfa_data["features"]), len(pfa_data["features"])\n'

In [30]:
'''
c=0
for i in ENG_msoa_data["features"]:
    if i["id"].startswith("W"):
        print(c)
        ENG_msoa_data["features"].pop(c)
    c += 1
len(ENG_msoa_data["features"]), len(msoa_data["features"])
'''

'\nc=0\nfor i in ENG_msoa_data["features"]:\n    if i["id"].startswith("W"):\n        print(c)\n        ENG_msoa_data["features"].pop(c)\n    c += 1\nlen(ENG_msoa_data["features"]), len(msoa_data["features"])\n'

In [31]:
'''
with open("ENG_pfa_data.geojson", "w") as file:
    json.dump(ENG_pfa_data, file)
with open("ENG_msoa_data.geojson", "w") as file:
    json.dump(ENG_msoa_data, file)
'''

'\nwith open("ENG_pfa_data.geojson", "w") as file:\n    json.dump(ENG_pfa_data, file)\nwith open("ENG_msoa_data.geojson", "w") as file:\n    json.dump(ENG_msoa_data, file)\n'

# Aggregating LSOA crime predictions to MSOA level

In [32]:
#English LSOA Crime Predictions
ENG_LSOA_pred = pd.read_csv("test_quantile_predictions.csv", usecols=["lsoa_code", "crime_type", "month", "q0.50"]).drop_duplicates().rename(columns={"lsoa_code":"LSOA21CD", "q0.50":"prediction"})

ENG_LSOA_pred_with_MSOA = pd.merge(ENG_LSOA_pred, LSOA_to_MSOA, on="LSOA21CD", how="inner")[["LSOA21CD", "crime_type", "month", "prediction", "MSOA21CD"]]

ENG_MSOA_pred = pd.DataFrame(ENG_LSOA_pred_with_MSOA.groupby(["MSOA21CD", "month", "crime_type"], as_index=False)["prediction"].sum())

# Aggregating MSOA level crime predictions to allocations buckets

In [33]:
ENG_MSOA_pred_with_buckets = pd.merge(ENG_MSOA_pred, bucket_to_crime, on="crime_type", how="inner")

ENG_MSOA_bucket_pred = pd.DataFrame(ENG_MSOA_pred_with_buckets.groupby(["MSOA21CD", "alloc_bucket", "month"], as_index=False)["prediction"].sum()).rename(columns={"alloc_bucket":"bucket"})

# Aggregating LSOA demographic data to MSOA level

In [34]:
'''
#English LSOA demographics data
ENG_LSOA_demog_data = pd.read_csv("msoa_data/lsoa_demographics_from_db_file.csv")

#English MSOA demographics data
#Initial dummy row
data = {"MSOA21CD":[0],
        "pop":[0],
        "working_pop":[0],
        "elderly_pop":[0],
        "child_pop":[0],
        "working_percent":[0],
        "elderly_percent":[0],
        "child_percent":[0],
        "econ_score":[0],
        "infrastructure_score":[0],
        "health_score":[0]
        }
ENG_MSOA_demog_data = pd.DataFrame(data)

#Aggregate LSOA data to MSOA level
for msoa in msoa_data["features"]:
    #Only use English MSOAs due to IMD data
    if not msoa["id"].startswith("W"):
        msoa_pop = 0
        msoa_working_pop = 0
        msoa_elderly_pop = 0
        msoa_child_pop = 0
        msoa_working_percent = 0
        msoa_elderly_percent = 0
        msoa_child_percent = 0
        msoa_econ_score = 0
        msoa_infrastructure_score = 0
        msoa_health_score = 0
        #Compute MSOA values using each LSOA in the MSOA
        LSOAs_in_msoa = LSOA_to_MSOA.loc[LSOA_to_MSOA["MSOA21CD"] == msoa["id"]]["LSOA21CD"]
        for lsoa in LSOAs_in_msoa:
            LSOA_row = ENG_LSOA_demog_data.loc[ENG_LSOA_demog_data["lsoa_code"] == lsoa]
            msoa_pop += LSOA_row["pop"].values[0]
            msoa_working_pop += LSOA_row["pop"].values[0] * LSOA_row["percent_working"].values[0]
            msoa_elderly_pop += LSOA_row["pop"].values[0] * LSOA_row["percent_old"].values[0]
            msoa_child_pop += LSOA_row["pop"].values[0] * LSOA_row["percent_child"].values[0]
            msoa_econ_score += LSOA_row["econ_score"].values[0]
            msoa_infrastructure_score += LSOA_row["infrastructure_score"].values[0]
            msoa_health_score += LSOA_row["health_score"].values[0]
        msoa_econ_score /= LSOAs_in_msoa.size
        msoa_infrastructure_score /= LSOAs_in_msoa.size
        msoa_health_score /= LSOAs_in_msoa.size
        msoa_working_percent = msoa_working_pop / msoa_pop
        msoa_elderly_percent = msoa_elderly_pop / msoa_pop
        msoa_child_percent = msoa_child_pop / msoa_pop
        #Package computed values
        data = {"MSOA21CD":[msoa["id"]],
                "pop":[msoa_pop],
                "working_pop":[msoa_working_pop],
                "elderly_pop":[msoa_elderly_pop],
                "child_pop":[msoa_child_pop],
                "working_percent":[msoa_working_percent],
                "elderly_percent":[msoa_elderly_percent],
                "child_percent":[msoa_child_percent],
                "econ_score":[msoa_econ_score],
                "infrastructure_score":[msoa_infrastructure_score],
                "health_score":[msoa_health_score]
               }
        msoa_dataframe = pd.DataFrame(data)
        #Merge the newly computed MSOA into the dataframe
        ENG_MSOA_demog_data = pd.concat([ENG_MSOA_demog_data, msoa_dataframe])
        
#Remove dummy row, fix indexing and export to CSV
ENG_MSOA_demog_data.index = range(ENG_MSOA_demog_data.shape[0])
ENG_MSOA_demog_data.drop(index=0, inplace=True)
ENG_MSOA_demog_data.index = range(ENG_MSOA_demog_data.shape[0])
ENG_MSOA_demog_data.to_csv("msoa_data/ENG_MSOA_demographic_data.csv", index=False)
'''

'\n#English LSOA demographics data\nENG_LSOA_demog_data = pd.read_csv("msoa_data/lsoa_demographics_from_db_file.csv")\n\n#English MSOA demographics data\n#Initial dummy row\ndata = {"MSOA21CD":[0],\n        "pop":[0],\n        "working_pop":[0],\n        "elderly_pop":[0],\n        "child_pop":[0],\n        "working_percent":[0],\n        "elderly_percent":[0],\n        "child_percent":[0],\n        "econ_score":[0],\n        "infrastructure_score":[0],\n        "health_score":[0]\n        }\nENG_MSOA_demog_data = pd.DataFrame(data)\n\n#Aggregate LSOA data to MSOA level\nfor msoa in msoa_data["features"]:\n    #Only use English MSOAs due to IMD data\n    if not msoa["id"].startswith("W"):\n        msoa_pop = 0\n        msoa_working_pop = 0\n        msoa_elderly_pop = 0\n        msoa_child_pop = 0\n        msoa_working_percent = 0\n        msoa_elderly_percent = 0\n        msoa_child_percent = 0\n        msoa_econ_score = 0\n        msoa_infrastructure_score = 0\n        msoa_health_sco

# Pre-compute choropleth data

In [35]:
#English PFA FTE resource choropleth data
pfa_resources = ENG_PFA_FTEs[["PFA21CD", "total_fte"]].set_index("PFA21CD").to_dict()["total_fte"]

#English MSOA FTE allocation choropleth data per bucket per month
ENG_msoa_allocations_b1_m1 = ENG_MSOA_alloc.loc[ENG_MSOA_alloc["bucket"] == "local_reassurance"].loc[ENG_MSOA_alloc["month"] == "2025-12"][["MSOA21CD", "total_alloc"]].set_index("MSOA21CD").to_dict()["total_alloc"]
ENG_msoa_allocations_b2_m1 = ENG_MSOA_alloc.loc[ENG_MSOA_alloc["bucket"] == "acquisitive_crime"].loc[ENG_MSOA_alloc["month"] == "2025-12"][["MSOA21CD", "total_alloc"]].set_index("MSOA21CD").to_dict()["total_alloc"]
ENG_msoa_allocations_b3_m1 = ENG_MSOA_alloc.loc[ENG_MSOA_alloc["bucket"] == "disorder_damage"].loc[ENG_MSOA_alloc["month"] == "2025-12"][["MSOA21CD", "total_alloc"]].set_index("MSOA21CD").to_dict()["total_alloc"]

ENG_msoa_allocations_b1_m2 = ENG_MSOA_alloc.loc[ENG_MSOA_alloc["bucket"] == "local_reassurance"].loc[ENG_MSOA_alloc["month"] == "2026-01"][["MSOA21CD", "total_alloc"]].set_index("MSOA21CD").to_dict()["total_alloc"]
ENG_msoa_allocations_b2_m2 = ENG_MSOA_alloc.loc[ENG_MSOA_alloc["bucket"] == "acquisitive_crime"].loc[ENG_MSOA_alloc["month"] == "2026-01"][["MSOA21CD", "total_alloc"]].set_index("MSOA21CD").to_dict()["total_alloc"]
ENG_msoa_allocations_b3_m2 = ENG_MSOA_alloc.loc[ENG_MSOA_alloc["bucket"] == "disorder_damage"].loc[ENG_MSOA_alloc["month"] == "2026-01"][["MSOA21CD", "total_alloc"]].set_index("MSOA21CD").to_dict()["total_alloc"]

ENG_msoa_allocations_b1_m3 = ENG_MSOA_alloc.loc[ENG_MSOA_alloc["bucket"] == "local_reassurance"].loc[ENG_MSOA_alloc["month"] == "2026-02"][["MSOA21CD", "total_alloc"]].set_index("MSOA21CD").to_dict()["total_alloc"]
ENG_msoa_allocations_b2_m3 = ENG_MSOA_alloc.loc[ENG_MSOA_alloc["bucket"] == "acquisitive_crime"].loc[ENG_MSOA_alloc["month"] == "2026-02"][["MSOA21CD", "total_alloc"]].set_index("MSOA21CD").to_dict()["total_alloc"]
ENG_msoa_allocations_b3_m4 = ENG_MSOA_alloc.loc[ENG_MSOA_alloc["bucket"] == "disorder_damage"].loc[ENG_MSOA_alloc["month"] == "2026-02"][["MSOA21CD", "total_alloc"]].set_index("MSOA21CD").to_dict()["total_alloc"]

#English MSOA crime prediction choropleth data per bucket per month
ENG_msoa_crime_predictions_b1_m1 = ENG_MSOA_bucket_pred.loc[ENG_MSOA_bucket_pred["bucket"] == "Neighbourhood Problem-Solving and Reassurance"].loc[ENG_MSOA_bucket_pred["month"] == "2025-12"][["MSOA21CD", "prediction"]].set_index("MSOA21CD").to_dict()["prediction"]
ENG_msoa_crime_predictions_b2_m1 = ENG_MSOA_bucket_pred.loc[ENG_MSOA_bucket_pred["bucket"] == "Acquisitive and Place-Based Prevention"].loc[ENG_MSOA_bucket_pred["month"] == "2025-12"][["MSOA21CD", "prediction"]].set_index("MSOA21CD").to_dict()["prediction"]
ENG_msoa_crime_predictions_b3_m1 = ENG_MSOA_bucket_pred.loc[ENG_MSOA_bucket_pred["bucket"] == "Harm, Disruption and Enforcement"].loc[ENG_MSOA_bucket_pred["month"] == "2025-12"][["MSOA21CD", "prediction"]].set_index("MSOA21CD").to_dict()["prediction"]

ENG_msoa_crime_predictions_b1_m2 = ENG_MSOA_bucket_pred.loc[ENG_MSOA_bucket_pred["bucket"] == "Neighbourhood Problem-Solving and Reassurance"].loc[ENG_MSOA_bucket_pred["month"] == "2026-01"][["MSOA21CD", "prediction"]].set_index("MSOA21CD").to_dict()["prediction"]
ENG_msoa_crime_predictions_b2_m2 = ENG_MSOA_bucket_pred.loc[ENG_MSOA_bucket_pred["bucket"] == "Acquisitive and Place-Based Prevention"].loc[ENG_MSOA_bucket_pred["month"] == "2026-01"][["MSOA21CD", "prediction"]].set_index("MSOA21CD").to_dict()["prediction"]
ENG_msoa_crime_predictions_b3_m2 = ENG_MSOA_bucket_pred.loc[ENG_MSOA_bucket_pred["bucket"] == "Harm, Disruption and Enforcement"].loc[ENG_MSOA_bucket_pred["month"] == "2026-01"][["MSOA21CD", "prediction"]].set_index("MSOA21CD").to_dict()["prediction"]

ENG_msoa_crime_predictions_b1_m3 = ENG_MSOA_bucket_pred.loc[ENG_MSOA_bucket_pred["bucket"] == "Neighbourhood Problem-Solving and Reassurance"].loc[ENG_MSOA_bucket_pred["month"] == "2026-02"][["MSOA21CD", "prediction"]].set_index("MSOA21CD").to_dict()["prediction"]
ENG_msoa_crime_predictions_b2_m3 = ENG_MSOA_bucket_pred.loc[ENG_MSOA_bucket_pred["bucket"] == "Acquisitive and Place-Based Prevention"].loc[ENG_MSOA_bucket_pred["month"] == "2026-02"][["MSOA21CD", "prediction"]].set_index("MSOA21CD").to_dict()["prediction"]
ENG_msoa_crime_predictions_b3_m3 = ENG_MSOA_bucket_pred.loc[ENG_MSOA_bucket_pred["bucket"] == "Harm, Disruption and Enforcement"].loc[ENG_MSOA_bucket_pred["month"] == "2026-02"][["MSOA21CD", "prediction"]].set_index("MSOA21CD").to_dict()["prediction"]

#English MSOA population choropleth data
ENG_msoa_pop = ENG_MSOA_demog_data[["MSOA21CD", "pop"]].set_index("MSOA21CD").to_dict()["pop"]

# Create basic PFA and MSOA layers

In [36]:
'''
pfa_layer = GeoJSON(data=pfa_data, 
    style={'color': 'black', 'fillColor': '#E0D071', 'opacity':0.25, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': '#b08a3e' , 'fillOpacity': 0.8}
)
msoa_layer = GeoJSON(data=msoa_data, 
    style={'color': 'black', 'fillColor': '#E0D071', 'opacity':0.25, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': '#b08a3e' , 'fillOpacity': 0.8}
)
'''

pfa_layer = GeoJSON(data=ENG_pfa_data,
    style={'color': 'black', 'opacity': 0.25, 'fillOpacity':0},
    hover_style={'opacity': 1, 'fillOpacity':0.25}
)

msoa_layer = GeoJSON(data=ENG_msoa_data,
    style={'color': 'black', 'opacity': 0.25, 'fillOpacity':0},
    hover_style={'opacity': 1, 'fillOpacity':0.25}
)

pfa_choropleth = Choropleth(
    geo_data=ENG_pfa_data,
    choro_data=pfa_resources,
    key_on="id",
    colormap=linear.YlGnBu_07
)

msoa_choropleth = Choropleth(
    geo_data=ENG_msoa_data,
    choro_data= ENG_msoa_allocations_b1_m1,
    key_on="id",
    colormap=linear.Oranges_08
)

# Creating the interactive visualisation

In [44]:
# ----------------#
#Create basic map#
#----------------#

center = [53,-2.5]
zoom = 7
m = Map(basemap=basemaps.CartoDB.Positron, center=center, zoom=zoom, zoom_control=False)

#----------------------------------#
#Create interactive control widgets#
#----------------------------------#

#Set up PFA information display HTML
html_pfa = HTML('''<h3><b>Hover over a Police Force to see further information!</b></h3>''')
html_pfa.layout.margin = '20px 20px 20px 20px'
pfa_control = WidgetControl(widget=html_pfa, position='topright')
html_pfa_click = HTML('''<h3><b>Click on a Police Force to see MSOA information!</b></h3>''')
html_pfa_click.layout.margin = '20px 20px 20px 20px'
pfa_click_control = WidgetControl(widget=html_pfa_click, position='topright')

#Set up MSOA information display HTMLs
html_msoa_demog = HTML('''<h3><b>Hover over an MSOA to see further information!!</b></h3>''')
html_msoa_demog.layout.margin = '20px 20px 20px 20px'
html_msoa_crime = HTML('''<h3><b>Hover over an MSOA to see further information!!</b></h3>''')
html_msoa_crime.layout.margin = '0px 0px 0px 0px'
html_msoa_alloc = HTML('''<h3><b>Hover over an MSOA to see further information!!</b></h3>''')
html_msoa_alloc.layout.margin = '0px 0px 0px 0px'

#Set up buttons that change what MSOA information is displayed
crime_button = Button(
    description='Crime Info',
    disabled=False,
    button_style='',
    tooltip="Displays Data for Crime Information",
)
allocation_button = Button(
    description='Allocation Info',
    disabled=False,
    button_style='',
    tooltip="Displays Data for Allocation Information",
)
demographic_button = Button(
    description='Demographic Info',
    disabled=False,
    button_style='',
    tooltip="Displays Data for Demographic Information",
)
#Package the buttons
info_buttons = HBox([allocation_button, crime_button, demographic_button])

#Set up prediction month selection
msoa_month_predict_select = IntSlider(
    value=1,
    min=1,
    max=3,
    step=1,
    description='',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
msoa_month_predict_select.layout.width = "50%"

#Set up button that refreshes the crime data when a new month is selected
display_month_crime_button = Button(
    description="Display",
    disabled=False,
    button_style='',
    tooltip="Displays Data for the Selected Month",
    icon="check"
)

#Set up button to pick which allocation bucket to use
msoa_bucket_pick = RadioButtons(
    options=['Neighbourhood Problem-Solving and Reassurance', 'Acquisitive and Place-Based Prevention', 'Harm, Disruption and Enforcement', 'All Specialisations'],
    value='Neighbourhood Problem-Solving and Reassurance',
    layout={'width': 'max-content'},
    description="Select which specialisation you wish to see information for:",
    disabled=False
)

#Package all crime and allocation information widgets
msoa_crime_box = VBox([VBox([Label(value="How many month(s) ahead do you wish to see crime information?"),
                             msoa_month_predict_select,
                             msoa_bucket_pick,
                             display_month_crime_button]),
                       html_msoa_crime])
msoa_alloc_box = VBox([VBox([Label(value="How many month(s) ahead do you wish to see allocation information?"),
                             msoa_month_predict_select,
                             msoa_bucket_pick,
                             display_month_crime_button]),
                       html_msoa_alloc])

#Package all MSOA information widgets
msoa_info = VBox([info_buttons, msoa_alloc_box])
msoa_info.layout.margin = "20px 20px 20px 20px"
msoa_info_control = WidgetControl(widget=msoa_info, position='topright')

#Set up button to return to PFA view in MSOA view
return_to_pfa_button = Button(
    description="Return to PFA view",
    disabled=False,
    button_style='',
    tooltip="Return to PFA view",
    icon="arrow-left"
)
return_control = WidgetControl(widget=return_to_pfa_button, position='topright')

fig = go.FigureWidget(data=[go.Pie(labels=[], values=[], textinfo="label+percent", showlegend=False)],
                      layout=go.Layout(width=500,
                                       height=500, 
                                       title="Hover over a Police Force!",
                                       title_font_size=20,
                                      )
                     )
plot_control = WidgetControl(widget=fig, position="bottomleft")

#-------------------------------------------------#
#Implement functions to update interactive widgets#
#-------------------------------------------------#

tab_open = ""

#Define PFA HTML update function
def update_pfa(**kwargs):
    html_pfa.value = '''
        <h3><b>Police Force: </b>{}</h3>
        <p>PFA Code: {}</p>
        <p>Available Neighbourhood Police (FTEs): <b>{}</b></p>
        <ul>
            <li>Of which Police Officers: <b>{}</b></li>
            <li>Of which PCSOs: <b>{}</b></li>
        <ul>
        '''.format(kwargs['properties']['PFA21NM'], kwargs['properties']['PFA21CD'], 
                   ENG_PFA_FTEs.loc[ENG_PFA_FTEs["PFA21CD"] == kwargs["id"]]["total_fte"].values[0],
                   ENG_PFA_FTEs.loc[ENG_PFA_FTEs["PFA21CD"] == kwargs["id"]]["officer_fte"].values[0],
                   ENG_PFA_FTEs.loc[ENG_PFA_FTEs["PFA21CD"] == kwargs["id"]]["pcso_fte"].values[0]
                  )
    fig.data[0].labels = ["PCSO FTEs", "Police Officer FTEs"]
    fig.data[0].values = ENG_PFA_FTEs.loc[ENG_PFA_FTEs["PFA21CD"] == kwargs["id"]][["pcso_fte", "officer_fte"]].squeeze(axis=0).values
    fig.layout.title = "Police Distribution in " + kwargs["properties"]["PFA21NM"]
    m.add(pfa_click_control)

#Define function to update the map when a PFA is clicked
def whenClicked_PFA(**kwargs):
    global tab_open
    MSOAs_in_PFA = MSOA_to_PFA.loc[MSOA_to_PFA["PFA21CD"] == kwargs["properties"]["PFA21CD"]]["MSOA21CD"]
    MSOAs_json = {
                  "type": "FeatureCollection", 
                  "crs": {"type": "name", "properties": {"name": "EPSG:4326"}},
                  "features": []
                 }
    for i in ENG_msoa_data["features"]:
        for j in MSOAs_in_PFA:
            if i["properties"]["MSOA21CD"] == j:
                MSOAs_json["features"].append(i)
                break
    msoa_layer.data = MSOAs_json
    msoa_choropleth.geo_data = MSOAs_json
    
    fig.data[0].labels = []
    fig.data[0].values = []
    fig.layout.title = "Hover over an MSOA!"

    tab_open = "alloc"
    
    m.substitute(pfa_layer, msoa_layer)
    m.substitute(pfa_choropleth, msoa_choropleth)
    m.substitute(pfa_click_control, msoa_info_control)
    m.add(return_control)
    m.center =  [kwargs['properties']['LAT'], kwargs['properties']['LONG']]
    m.zoom = 9.25


#Define MSOA HTML update function
def update_msoa(**kwargs):
    global tab_open
    month = ""
    #IF-ELIF CLAUSES DECIDING WHAT MONTH PREDICTION/ALLOCATION TO DISPLAY
    if msoa_month_predict_select.value == 1:  
        month = "2025-12"
    elif msoa_month_predict_select.value == 2:
        month = "2026-01"
    elif msoa_month_predict_select.value == 3:
        month = "2026-02"
    #ALLOCATION BUCKET WE ARE DISPLAYING
    bucket = msoa_bucket_pick.value
    alloc_bucket = ""
    if bucket == "Neighbourhood Problem-Solving and Reassurance":
        alloc_bucket = "local_reassurance"
    elif bucket == "Acquisitive and Place-Based Prevention":
        alloc_bucket = "acquisitive_crime"
    elif bucket == "Harm, Disruption and Enforcement":
        alloc_bucket = "disorder_damage"
    elif bucket == "All Specialisations":
        alloc_bucket = "all"

    pred_chosen_msoa_month = ENG_MSOA_pred.loc[ENG_MSOA_pred["MSOA21CD"] == kwargs["id"]].loc[ENG_MSOA_pred["month"] == month]
    #Updating HTMLs
    if bucket == "Neighbourhood Problem-Solving and Reassurance":
        if tab_open == "crime":
            fig.data[0].values = [pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Anti-social behaviour"]["prediction"].squeeze(axis=0)]
            fig.data[0].labels = ["Anti-social behaviour"]
            fig.layout.title = "Crime Prediction Distribution in " + kwargs["properties"]["MSOA21NM"]
        html_msoa_crime.value = '''
            <h3><b>MSOA: </b>{}</h3>
            <p>MSOA Code: {}</p>
            <p>Predicted Crime Counts per Relevant Category:</p>
            <ul>
                <li>Anti-Social Behaviour: <b>{}</b></li>
            </ul>
            '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'], 
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Anti-social behaviour"]["prediction"].values[0], 3)
                      )
    elif bucket == "Acquisitive and Place-Based Prevention":
        if tab_open == "crime":
            fig.data[0].values = pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"].isin(["Property theft", "Criminal damage and arson", "Vehicle crime", "Burglary"])]["prediction"].squeeze(axis=0).values
            fig.data[0].labels = ["Burglary", "Criminal damage and arson", "Property theft", "Vehicle crime"]
            fig.layout.title = "Crime Prediction Distribution in " + kwargs["properties"]["MSOA21NM"]
        html_msoa_crime.value = '''
            <h3><b>MSOA: </b>{}</h3>
            <p>MSOA Code: {}</p>
            <p>Predicted Crime Counts per Relevant Category:</p>
            <ul>
                <li>Property Theft: <b>{}</b></li>
                <li>Criminal Damage and Arson: <b>{}</b></li>
                <li>Vehicle Crime: <b>{}</b></li>
                <li>Burglary: <b>{}</b></li>
            </ul>
            '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'], 
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Property theft"]["prediction"].values[0], 3), 
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Criminal damage and arson"]["prediction"].values[0], 3),
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Vehicle crime"]["prediction"].values[0], 3),
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Burglary"]["prediction"].values[0], 3)
                      )
    elif bucket == "Harm, Disruption and Enforcement":
        if tab_open == "crime":
            fig.data[0].values = pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"].isin(["Violence", "Disruptive crimes"])]["prediction"].squeeze(axis=0).values
            fig.data[0].labels = ["Disruptive crimes", "Violence"]
            fig.layout.title = "Crime Prediction Distribution in " + kwargs["properties"]["MSOA21NM"]
        html_msoa_crime.value = '''
            <h3><b>MSOA: </b>{}</h3>
            <p>MSOA Code: {}</p>
            <p>Predicted Crime Counts per Relevant Category:</p>
            <ul>
                <li>Violence: <b>{}</b></li>
                <li>Disruptive Crimes: <b>{}</b></li>
            </ul>
            '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'], 
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Violence"]["prediction"].values[0], 3),
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Disruptive crimes"]["prediction"].values[0], 3)
                      )
    elif bucket == "All Specialisations":
        if tab_open == "crime":
            fig.data[0].values = pred_chosen_msoa_month["prediction"].squeeze(axis=0).values
            fig.data[0].labels = ["Anti-social Behaviour", "Burglary", "Criminal Damage and Arson", "Disruptive Crimes", "Property Theft", "Vehicle Crime", "Violence"]
            fig.layout.title = "Crime Prediction Distribution in " + kwargs["properties"]["MSOA21NM"]
        html_msoa_crime.value = '''
            <h3><b>MSOA: </b>{}</h3>
            <p>MSOA Code: {}</p>
            <p>Predicted Crime Counts per Relevant Category:</p>
            <ul>
                <li>Anti-Social Behaviour: <b>{}</b></li>
                <li>Property Theft: <b>{}</b></li>
                <li>Criminal Damage and Arson: <b>{}</b></li>
                <li>Vehicle Crime: <b>{}</b></li>
                <li>Burglary: <b>{}</b></li>
                <li>Violence: <b>{}</b></li>
                <li>Disruptive Crimes: <b>{}</b></li>
            </ul>
            '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'], 
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Anti-social behaviour"]["prediction"].values[0], 3),
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Property theft"]["prediction"].values[0], 3), 
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Criminal damage and arson"]["prediction"].values[0], 3),
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Vehicle crime"]["prediction"].values[0], 3),
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Burglary"]["prediction"].values[0], 3),
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Violence"]["prediction"].values[0], 3),
                       round(pred_chosen_msoa_month.loc[pred_chosen_msoa_month["crime_type"] == "Disruptive crimes"]["prediction"].values[0], 3)
                      )
        
    if alloc_bucket == "all":
        alloc_pie_values = ENG_MSOA_alloc.loc[ENG_MSOA_alloc["MSOA21CD"] == kwargs["id"]].loc[ENG_MSOA_alloc["month"] == month]["total_alloc"].squeeze(axis=0).values
        alloc_pie_labels = ["Acquisitive", "Harm, Disruption", "Reassurance"]
        html_msoa_alloc.value = '''
            <h3><b>MSOA: </b>{}</h3>
            <p>MSOA Code: {}</p>
            <p>Recommended allocation of officers:</p>
            <ul>
                <li>Reassurance: <b>{}</b></li>
                <li>Acquisitive: <b>{}</b></li>
                <li>Harm, Disruption: <b>{}</b></li>
            </ul>
            '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'], 
                       round(ENG_MSOA_alloc.loc[ENG_MSOA_alloc["MSOA21CD"] == kwargs["id"]].loc[ENG_MSOA_alloc["month"] == month].loc[ENG_MSOA_alloc["bucket"] == "local_reassurance"]["total_alloc"].values[0], 3),
                       round(ENG_MSOA_alloc.loc[ENG_MSOA_alloc["MSOA21CD"] == kwargs["id"]].loc[ENG_MSOA_alloc["month"] == month].loc[ENG_MSOA_alloc["bucket"] == "acquisitive_crime"]["total_alloc"].values[0], 3),
                       round(ENG_MSOA_alloc.loc[ENG_MSOA_alloc["MSOA21CD"] == kwargs["id"]].loc[ENG_MSOA_alloc["month"] == month].loc[ENG_MSOA_alloc["bucket"] == "disorder_damage"]["total_alloc"].values[0], 3)
                      )
    else:
        alloc_pie_values = [ENG_MSOA_alloc.loc[ENG_MSOA_alloc["MSOA21CD"] == kwargs["id"]].loc[ENG_MSOA_alloc["month"] == month].loc[ENG_MSOA_alloc["bucket"] == alloc_bucket]["total_alloc"].squeeze(axis=0)]
        alloc_pie_labels = [bucket]
        html_msoa_alloc.value = '''
            <h3><b>MSOA: </b>{}</h3>
            <p>MSOA Code: {}</p>
            <p>Recommended allocation of officers: <b>{}</b></p>
            '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'], 
                       round(ENG_MSOA_alloc.loc[ENG_MSOA_alloc["MSOA21CD"] == kwargs["id"]].loc[ENG_MSOA_alloc["month"] == month].loc[ENG_MSOA_alloc["bucket"] == alloc_bucket]["total_alloc"].values[0], 3)
                      )
    if tab_open == "alloc":
        fig.data[0].values = alloc_pie_values
        fig.data[0].labels = alloc_pie_labels
        fig.layout.title = "Police Allocation Distribution in " + kwargs["properties"]["MSOA21NM"]

    if tab_open == "demog":
        fig.data[0].values = ENG_MSOA_demog_data.loc[ENG_MSOA_demog_data["MSOA21CD"] == kwargs["id"]][["working_pop", "elderly_pop", "child_pop"]].squeeze(axis=0).values
        fig.data[0].labels = ["Working Age", "Elderly", "Children"]
        fig.layout.title = "Demographic Distribution in " + kwargs["properties"]["MSOA21NM"]

    msoa_demog_row = ENG_MSOA_demog_data.loc[ENG_MSOA_demog_data["MSOA21CD"] == kwargs["id"]]
    html_msoa_demog.value = '''
        <h3><b>MSOA: </b>{}</h3>
        <p>MSOA Code: {}</p>
        <p>Population Estimates: <b>~{}</b></p>
        <ul>
            <li>Of which working age: <b>~{}</b> (~{}%)</li>
            <li>Of which children: <b>~{}</b> (~{}%)</li>
            <li>Of which elderly: <b>~{}</b> (~{}%)</li>
        </ul>
        <p>Average Demographic Scores across Constituient LSOAs:</p>
        <ul>
            <li>Economy Score: <b>{}b</b></li>
            <li>Infrastructure Score: <b>{}</b></li>
            <li>Health Score: <b>{}</b></li>
        </ul>
        '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'], 
                   round(msoa_demog_row["pop"].values[0]),
                   round(msoa_demog_row["working_pop"].values[0]), round(msoa_demog_row["working_percent"].values[0]*100),
                   round(msoa_demog_row["child_pop"].values[0]), round(msoa_demog_row["child_percent"].values[0]*100),
                   round(msoa_demog_row["elderly_pop"].values[0]), round(msoa_demog_row["elderly_percent"].values[0]*100),
                   round(msoa_demog_row["econ_score"].values[0], 3),
                   round(msoa_demog_row["infrastructure_score"].values[0], 3),
                   round(msoa_demog_row["health_score"].values[0], 3)
                  )
    html_msoa_demog.layout.margin = "0px 0px 0px 0px"
    
#Define MSOA info change button function
def msoa_info_change(button_instance):
    global tab_open
    month = ""
    #IF-ELIF CLAUSES DECIDING WHAT MONTH PREDICTION/ALLOCATION TO DISPLAY
    if msoa_month_predict_select.value == 1:  
        month = "2025-12"
    elif msoa_month_predict_select.value == 2:
        month = "2026-01"
    elif msoa_month_predict_select.value == 3:
        month = "2026-02"
    #ALLOCATION BUCKET WE ARE DISPLAYING
    bucket = msoa_bucket_pick.value
    alloc_bucket = ""
    if bucket == "Neighbourhood Problem-Solving and Reassurance":
        alloc_bucket = "local_reassurance"
    elif bucket == "Acquisitive and Place-Based Prevention":
        alloc_bucket = "acquisitive_crime"
    elif bucket == "Harm, Disruption and Enforcement":
        alloc_bucket = "disorder_damage"
    elif bucket == "All Specialisations":
        alloc_bucket = "all"
    
    if button_instance.description == 'Crime Info':
        tab_open = "crime"
        msoa_info.children = [info_buttons, msoa_crime_box]
        if bucket == "All Specialisations":
            msoa_choropleth.choro_data = ENG_MSOA_bucket_pred.loc[ENG_MSOA_bucket_pred["month"] == month].groupby(["MSOA21CD"], as_index=False)["prediction"].sum().set_index("MSOA21CD").to_dict()["prediction"]
        else:
            msoa_choropleth.choro_data = ENG_MSOA_bucket_pred.loc[ENG_MSOA_bucket_pred["bucket"] == bucket].loc[ENG_MSOA_bucket_pred["month"] == month][["MSOA21CD", "prediction"]].set_index("MSOA21CD").to_dict()["prediction"]
    elif button_instance.description == 'Allocation Info':
        tab_open = "alloc"
        msoa_info.children = [info_buttons, msoa_alloc_box]
        if alloc_bucket == "all":
            msoa_choropleth.choro_data = ENG_MSOA_alloc.loc[ENG_MSOA_alloc["month"] == month].groupby(["MSOA21CD"], as_index=False)["total_alloc"].sum().set_index("MSOA21CD").to_dict()["total_alloc"]
        else:
            msoa_choropleth.choro_data =  ENG_MSOA_alloc.loc[ENG_MSOA_alloc["bucket"] == alloc_bucket].loc[ENG_MSOA_alloc["month"] == month][["MSOA21CD", "total_alloc"]].set_index("MSOA21CD").to_dict()["total_alloc"]
    elif button_instance.description == 'Demographic Info':
        tab_open = "demog"
        msoa_info.children = [info_buttons, html_msoa_demog]
        msoa_choropleth.choro_data = ENG_MSOA_demog_data[["MSOA21CD", "pop"]].set_index("MSOA21CD").to_dict()["pop"]
    print(len(msoa_choropleth.choro_data))
    msoa_info_control.widget=msoa_info
    m.remove(msoa_layer)
    m.add(msoa_layer)
    
    fig.data[0].labels = []
    fig.data[0].values = []
    fig.layout.title = "Hover over an MSOA!"

#Define return button function
def back_to_pfa(button_instance):
    html_msoa_demog.value = '''<h3><b>Hover over an MSOA to see further information!!</b></h3>'''
    html_msoa_crime.value = '''<h3><b>Hover over an MSOA to see further information!!</b></h3>'''
    html_msoa_alloc.value = '''<h3><b>Hover over an MSOA to see further information!!</b></h3>'''
    html_pfa.value = '''<h3><b>Hover over a Police Force to see further information!</b></h3>'''

    fig.data[0].labels = []
    fig.data[0].values = []
    fig.layout.title = "Hover over a Police Force!"
    
    m.substitute(msoa_layer, pfa_layer)
    m.substitute(msoa_choropleth, pfa_choropleth)
    m.remove(msoa_info_control)
    m.remove(return_control)
    m.center =  center
    m.zoom = zoom   
    
#----------------------------------#
#Attach update functions to widgets#
#----------------------------------#

pfa_layer.on_click(whenClicked_PFA)
pfa_layer.on_hover(update_pfa)

msoa_layer.on_hover(update_msoa)

return_to_pfa_button.on_click(back_to_pfa)
display_month_crime_button.on_click(update_msoa)
crime_button.on_click(msoa_info_change)
allocation_button.on_click(msoa_info_change)
demographic_button.on_click(msoa_info_change)

#----------------------------------------#
#Add default elements and display the map#
#----------------------------------------#

m.add(pfa_choropleth)
m.add(pfa_layer)
m.add(ZoomControl(position="bottomright"))
m.add(FullScreenControl(position="bottomright"))
m.add(pfa_control)
m.add(plot_control)

m

Map(center=[53, -2.5], controls=(AttributionControl(options=['position', 'prefix'], position='bottomright'), Z…